# Day 6 Project Solution: Text Transformation Prompt Library

Five named transformations — `summarize`, `expand`, `change_tone`, `extract_keywords`,
and `eli5` — each engineered with the full Day 6 toolkit: four-part anatomy, role
system prompts, few-shot examples, and format constraints, all wired together through
a single `apply_transform(name, text)` dispatcher.

## Setup

In [ ]:
import ollama

MODEL = "llama3.2"

# All five transforms run on this paragraph
SAMPLE = (
    "The human brain contains roughly 86 billion neurons, each connected to "
    "thousands of others via synapses. When you learn something new, repeated "
    "activation of a neural pathway strengthens the synaptic connections along "
    "that path — a process called synaptic plasticity. This is why practice "
    "makes permanent: the more you repeat an action or recall a fact, the "
    "faster and more reliably that circuit fires."
)

## Prompt-Builder Functions

Each builder owns the prompt structure (Lesson 1 anatomy + Lesson 2 role +
Lesson 3 few-shot + Lesson 4 format constraint) and returns a `list[dict]`.
No model calls here — the dispatcher calls the model.

In [ ]:
# --- TRANSFORM 1: summarize ---
# Lesson 1 (anatomy) + Lesson 2 (role) + Lesson 4 (format constraint)

def build_summarize_prompt(text: str) -> list[dict]:
    """Condense text to one short paragraph of at most 40 words.

    Args:
        text: The source text to summarize.

    Returns:
        messages list ready for ollama.chat().
    """
    return [
        {
            # Lesson 2: role sets the model's persona and tone
            "role": "system",
            "content": (
                "You are a concise technical writer. "
                "Your job is to distill text to its essential meaning "
                "without losing accuracy."
            ),
        },
        {
            "role": "user",
            "content": (
                # Lesson 1: instruction + context already live in the system prompt;
                # input and output format go here in the user message
                f"{text}\n\n"
                "Summarize the above in at most 40 words. "
                # Lesson 4: precise format constraint — prohibit preamble/heading
                "Write one paragraph. Do not add a heading or any introductory phrase."
            ),
        },
    ]

In [ ]:
# --- TRANSFORM 2: expand ---
# Lesson 1 (anatomy) + Lesson 2 (role) + Lesson 4 (format constraint)

def build_expand_prompt(text: str) -> list[dict]:
    """Elaborate on the text with two paragraphs and concrete examples.

    Args:
        text: The source text to expand.

    Returns:
        messages list ready for ollama.chat().
    """
    return [
        {
            # Lesson 2: science-communicator role brings appropriate vocabulary
            "role": "system",
            "content": (
                "You are a science communicator who makes complex ideas vivid "
                "and accessible. You always ground abstract concepts in concrete, "
                "real-world examples."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{text}\n\n"
                "Expand the above into exactly two paragraphs. "
                # Lesson 4: structural detail — two paragraphs, facts preserved, example added
                "Keep all original facts. Add at least one concrete real-world example. "
                "Do not add a heading or any introductory phrase."
            ),
        },
    ]

In [ ]:
# --- TRANSFORM 3: change_tone (formal) ---
# Lesson 1 (anatomy) + Lesson 2 (role with parameterised tone) + Lesson 4 (sentinel)

def build_change_tone_prompt(text: str, tone: str = "formal") -> list[dict]:
    """Rewrite text in the specified tone while preserving all meaning.

    Args:
        text: The original text to rewrite.
        tone: Target tone, e.g. 'formal', 'casual', 'persuasive'.

    Returns:
        messages list ready for ollama.chat().
    """
    return [
        {
            # Lesson 2: embed tone parameter directly in the role description
            "role": "system",
            "content": (
                f"You are a professional editor specialising in {tone} writing. "
                "Rewrite text in the requested tone while preserving every fact. "
                "Do not add or remove information."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Rewrite the following text in a {tone} tone:\n\n{text}\n\n"
                "Keep approximately the same length as the original. "
                "Do not include any preamble or explanation.\n"
                # Lesson 4: soft sentinel — 'Rewritten version:' steers opening token
                "Rewritten version:"
            ),
        },
    ]

In [ ]:
# --- TRANSFORM 4: extract_keywords ---
# Lesson 1 (anatomy) + Lesson 2 (role) + Lesson 3 (few-shot) + Lesson 4 (format)

# Lesson 3: store examples as (input, output) tuples — easy to extend or reorder
_KEYWORD_EXAMPLES: list[tuple[str, str]] = [
    (
        # Example 1: biology text -> numbered keyword list
        "Photosynthesis is the process by which plants use sunlight, water, and "
        "carbon dioxide to produce glucose and oxygen. Chlorophyll in the leaves "
        "absorbs light energy to drive the chemical reactions.",
        "1. photosynthesis\n2. chlorophyll\n3. glucose\n4. sunlight\n5. carbon dioxide",
    ),
    (
        # Example 2: tech text -> numbered keyword list
        "Machine learning models are trained on large datasets to identify patterns "
        "and make predictions. Gradient descent is the optimisation algorithm that "
        "adjusts model weights to minimise the loss function during training.",
        "1. machine learning\n2. training data\n3. gradient descent\n4. loss function\n5. model weights",
    ),
]


def build_extract_keywords_prompt(text: str) -> list[dict]:
    """Extract 3-5 key terms as a numbered list, demonstrated with few-shot examples.

    Args:
        text: The text to extract keywords from.

    Returns:
        messages list ready for ollama.chat().
    """
    # Lesson 2: specialist role clarifies what 'keyword' means in this context
    messages: list[dict] = [
        {
            "role": "system",
            "content": (
                "You are a keyword extraction specialist. "
                "Identify the most important technical or domain-specific terms "
                "from the provided text. Reply with a numbered list only."
            ),
        }
    ]

    # Lesson 3: insert examples as alternating user/assistant turns before real input
    for user_ex, assistant_ex in _KEYWORD_EXAMPLES:
        messages.append({"role": "user",      "content": user_ex})
        messages.append({"role": "assistant", "content": assistant_ex})

    # Real input goes last; Lesson 4: explicit format constraint locks the shape
    messages.append({
        "role": "user",
        "content": (
            f"{text}\n\n"
            "Extract 3 to 5 key terms as a numbered list. "
            "One term per line. No explanation, no heading, no extra text."
        ),
    })
    return messages

In [ ]:
# --- TRANSFORM 5: eli5 ---
# Lesson 1 (anatomy) + Lesson 2 (role) + Lesson 4 (format constraint)

def build_eli5_prompt(text: str) -> list[dict]:
    """Explain the text like the reader is 5 years old — 2 sentences, no jargon.

    Args:
        text: The text to explain simply.

    Returns:
        messages list ready for ollama.chat().
    """
    return [
        {
            # Lesson 2: very specific role — young-child teacher, not generic explainer
            "role": "system",
            "content": (
                "You are a patient, warm teacher explaining ideas to a 5-year-old. "
                "You never use technical jargon. "
                "You always use a simple everyday analogy to make the idea concrete."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{text}\n\n"
                "Explain this to a 5-year-old in exactly 2 sentences. "
                # Lesson 4: hard format constraint + prohibition of jargon
                "Use at least one simple analogy. "
                "Do not use any scientific or technical terms. "
                "Do not add a heading or any introductory phrase."
            ),
        },
    ]

## Prompt Library and Dispatcher

Lesson 5 pattern: `PROMPT_LIBRARY` is a single dict of named builder functions.
`apply_transform` is the one call site — it hides all prompt engineering from callers.

In [ ]:
# Lesson 5: one dict — names map to builder functions
PROMPT_LIBRARY: dict = {
    "summarize":        build_summarize_prompt,
    "expand":           build_expand_prompt,
    "change_tone":      build_change_tone_prompt,
    "extract_keywords": build_extract_keywords_prompt,
    "eli5":             build_eli5_prompt,
}


def apply_transform(name: str, text: str) -> tuple[str, str]:
    """Build the prompt for the named transform and call the model.

    Args:
        name: Key in PROMPT_LIBRARY.
        text: The input text to transform.

    Returns:
        (prompt_preview, model_response)
        prompt_preview — the last user message content (shows what was sent).
        model_response — the model's reply as a plain string.

    Raises:
        KeyError: if name is not registered in PROMPT_LIBRARY.
    """
    if name not in PROMPT_LIBRARY:
        raise KeyError(f"Unknown transform {name!r}. Available: {list(PROMPT_LIBRARY)}")

    builder = PROMPT_LIBRARY[name]
    messages = builder(text)         # pure construction — no model call

    # Last user message is the prompt preview; lets the caller inspect what was sent
    prompt_preview = next(
        (m["content"] for m in reversed(messages) if m["role"] == "user"),
        "",
    )

    response = ollama.chat(model=MODEL, messages=messages)
    model_response = response["message"]["content"].strip()

    return prompt_preview, model_response

## Run All Five Transforms

In [ ]:
# Call every registered transform on the sample paragraph and print results

results = {}   # store responses for the gate check below

for transform_name in PROMPT_LIBRARY:
    prompt_preview, response_text = apply_transform(transform_name, SAMPLE)
    results[transform_name] = response_text

    print(f"{'=' * 60}")
    print(f"TRANSFORM: {transform_name.upper()}")
    print(f"{'=' * 60}")
    print("[PROMPT SENT]")
    # Trim long prompts to first 300 chars for readability
    preview = prompt_preview if len(prompt_preview) <= 300 else prompt_preview[:300] + " ..."
    print(preview)
    print()
    print("[MODEL RESPONSE]")
    print(response_text)
    print()

## Gate Check

In [ ]:
# Verify every transform ran and returned a non-empty response

expected_transforms = {"summarize", "expand", "change_tone", "extract_keywords", "eli5"}

assert set(results.keys()) == expected_transforms, (
    f"Missing transforms: {expected_transforms - set(results.keys())}"
)

for name, response in results.items():
    assert isinstance(response, str) and len(response) > 0, (
        f"Transform '{name}' returned an empty response"
    )

# Verify the library has exactly five entries
assert len(PROMPT_LIBRARY) == 5, (
    f"Expected 5 transforms in PROMPT_LIBRARY, got {len(PROMPT_LIBRARY)}"
)

# Verify extract_keywords builder includes few-shot examples (at least 2 assistant turns)
_kw_msgs = build_extract_keywords_prompt("test")
_assistant_turns = [m for m in _kw_msgs if m["role"] == "assistant"]
assert len(_assistant_turns) >= 2, (
    "extract_keywords builder must include at least 2 few-shot example (assistant) turns"
)

# Verify every builder's first message has role 'system'
for _name, _builder in PROMPT_LIBRARY.items():
    _msgs = _builder("sample text")
    assert _msgs[0]["role"] == "system", (
        f"Builder '{_name}': first message must be role='system'"
    )

print("All 5 transforms complete.")
print("Day 6 deliverable: five reliably formatted LLM outputs from one reusable prompt library.")